In [2]:
import darts
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

In [12]:
import polars as pl

# 1. Scan
lf = pl.scan_csv("data.csv")

# 2. Transformation
processed_lf = (
    lf.with_columns([
        # Use str.slice to handle the " UTC" suffix efficiently
        pl.col("timedate")
          .str.slice(0, 19) 
          .str.to_datetime("%Y-%m-%d %H:%M:%S")
          .dt.truncate("1h")
          .alias("hour")
    ])
    .group_by(["deviceId", "period", "hour"])
    .agg([
        # Calculate mean for temperature columns t1 through t13
        # This regex ensures we only pick columns that are 't' followed by digits
        pl.col("^t\d+$").mean(), 
        
        # Other metrics
        pl.col("x1").mean(),
        pl.col("x2").mean(),
        pl.col("t1").min().alias("t1_min"),
        pl.col("t1").max().alias("t1_max"),
        pl.col("x3").first(),
        pl.col("deviceType").first()
    ])
    .sort(["deviceId", "hour"])
)

# 3. Single-pass execution
print("Starting aggregation (streaming 10GB)...")
final_df = processed_lf.collect(streaming=True)

print("Splitting and saving files...")
for p in ["train", "test", "valid"]:
    subset = final_df.filter(pl.col("period") == p)
    if not subset.is_empty():
        subset.write_csv(f"{p}_hourly.csv")
        print(f"✅ Saved {p}_hourly.csv ({len(subset)} rows)")

<>:20: SyntaxWarning: invalid escape sequence '\d'
<>:20: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_23663/2720612027.py:20: SyntaxWarning: invalid escape sequence '\d'
  pl.col("^t\d+$").mean(),
/tmp/ipykernel_23663/2720612027.py:35: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  final_df = processed_lf.collect(streaming=True)


Starting aggregation (streaming 10GB)...
Splitting and saving files...
✅ Saved train_hourly.csv (2905879 rows)
✅ Saved test_hourly.csv (1660460 rows)
✅ Saved valid_hourly.csv (841834 rows)


In [3]:
devices = pd.read_csv('devices.csv')
devices

,latitude,longitude,deviceId
0,50.0,18.3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...
1,53.5,21.1,005767201ec5d7c3336b3b4d1ffa8a72e7ca1ecdaac30f...
2,52.9,18.1,01668c64ccc16c506a7c1a5c032e2eb5e2de48ecb284f2...
3,52.5,17.7,01bf745bf2df0312bd5ff2234c0e9dedc39ad0bac9bcfc...
4,50.7,16.7,02e4ad5d8d0016d35a003ea6df7e10fe27093aba81c64a...
...,...,...,...
595,52.4,14.6,fbdf266dd372334fcd865972f980574608f7fc122ecf6c...
596,54.6,18.1,fc376d9b600a622120a38da673f71d121b850055851e26...
597,51.9,19.6,fd2d77f969dd205ffe471e2349d3d2f54dc6e30a7d2ccd...
598,51.5,21.5,fdd1157e24eae9617881107d653c75ee16234d3bb7d84f...


In [4]:
df = pd.read_csv('train_hourly.csv')


def augment_df(df, dropId = True):
    df = df.merge(devices, on="deviceId", how="left")

    df["timedate"] = pd.to_datetime(df["hour"], utc=True)

    # encode time features
    df["month"] = df["timedate"].dt.month
    df["hour"] = df["timedate"].dt.hour

    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    df.drop(columns=["timedate", "hour", "month", "period"], inplace=True)

    if dropId:
        df.drop(columns=["deviceId"], inplace=True) 

    return df

train_df = augment_df(pd.read_csv('train_hourly.csv'))
valid_df = augment_df(pd.read_csv('valid_hourly.csv'), dropId=False)
test_df = augment_df(pd.read_csv('test_hourly.csv'), dropId=False)

In [36]:
train_df.head()

,t1,t2,t3,t4,t5,t6,t7,t8,t9,t10,...,t1_min,t1_max,x3,deviceType,latitude,longitude,month_sin,month_cos,hour_sin,hour_cos
0,0.290000,0.05,0.0,0.372500,0.446667,0.438333,0.200000,0.499167,0.394167,0.210000,...,0.29,0.29,8,19,50.0,18.3,-0.866025,0.5,0.000000,1.000000
1,0.290000,0.05,0.0,0.418333,0.464167,0.452500,0.200000,0.514167,0.408333,0.210000,...,0.29,0.29,8,19,50.0,18.3,-0.866025,0.5,0.258819,0.965926
2,0.290000,0.05,0.0,0.447500,0.477500,0.450000,0.205833,0.530833,0.390000,0.210000,...,0.29,0.29,8,19,50.0,18.3,-0.866025,0.5,0.500000,0.866025
3,0.286667,0.05,0.0,0.392500,0.459167,0.452500,0.210000,0.506667,0.403333,0.210000,...,0.28,0.29,8,19,50.0,18.3,-0.866025,0.5,0.707107,0.707107
4,0.280000,0.05,0.0,0.445000,0.473333,0.456667,0.210000,0.521667,0.415000,0.208333,...,0.28,0.28,8,19,50.0,18.3,-0.866025,0.5,0.866025,0.500000


In [41]:
train_df.shape

(2905879, 25)

In [26]:
split_point = int(0.8 * train_df.shape[0])
train_test_df = train_df[:split_point]
train_valid_df = train_df[split_point:]

In [37]:
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

target = "x2"

X = train_df.drop(columns=[target])
y = train_df[target]

split_idx = int(len(train_df) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

# Convert to DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# Model parameters
params = {
    "objective": "reg:squarederror",
    "eval_metric": "mae",
    "max_depth": 6,
    "eta": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
}

# Train model
model = xgb.train(
    params,
    dtrain,
    num_boost_round=200,
    evals=[(dtrain, "train"), (dtest, "test")],
    early_stopping_rounds=20,
)

# Predict
preds = model.predict(dtest)

# Evaluate
rmse = mean_absolute_error(y_test, preds)
print("MAE:", rmse)

[0]	train-mae:0.13388	test-mae:0.12638
[1]	train-mae:0.12441	test-mae:0.11733
[2]	train-mae:0.11546	test-mae:0.10898
[3]	train-mae:0.10804	test-mae:0.10213
[4]	train-mae:0.09914	test-mae:0.09390
[5]	train-mae:0.09134	test-mae:0.08678
[6]	train-mae:0.08443	test-mae:0.08056
[7]	train-mae:0.07835	test-mae:0.07510
[8]	train-mae:0.07341	test-mae:0.07080
[9]	train-mae:0.06985	test-mae:0.06774
[10]	train-mae:0.06608	test-mae:0.06462
[11]	train-mae:0.06238	test-mae:0.06136
[12]	train-mae:0.05914	test-mae:0.05852
[13]	train-mae:0.05639	test-mae:0.05631
[14]	train-mae:0.05386	test-mae:0.05419
[15]	train-mae:0.05166	test-mae:0.05232
[16]	train-mae:0.04971	test-mae:0.05077
[17]	train-mae:0.04799	test-mae:0.04940
[18]	train-mae:0.04635	test-mae:0.04806
[19]	train-mae:0.04498	test-mae:0.04702
[20]	train-mae:0.04372	test-mae:0.04603
[21]	train-mae:0.04250	test-mae:0.04505
[22]	train-mae:0.04131	test-mae:0.04425
[23]	train-mae:0.04043	test-mae:0.04354
[24]	train-mae:0.03952	test-mae:0.04281
[25]	train

In [1]:
target = "x2"

full_test = pd.concat([valid_df, test_df], ignore_index=True)
feature_cols = [c for c in full_test.columns if c not in ['x2', 'deviceId', 'year', 'month']]

X_submit = full_test[feature_cols]
dsubmit = xgb.DMatrix(X_submit)
full_test['prediction'] = model.predict(dsubmit)

submission = (
    full_test.groupby(['deviceId', 'year', 'month'])['prediction']
    .mean()
    .reset_index()
)

# 5. Format and Save
# Ensure the columns match the submission requirements exactly
submission = submission[['deviceId', 'year', 'month', 'prediction']]
submission.to_csv("submission.csv", index=False)

print(f"Submission file created with {len(submission)} rows.")
print(submission.head())

SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='? (3616084393.py, line 14)